# Explicit checking

Construction, loading, and parsing validate structure. `.check()` adds profile, history, and optional argument-schema checks. `ok` means no errors; `complete` means no requested applicable checks lacked evidence. Neither checking nor loading repairs data.


In [ ]:
from apertus_common import (
    ArgumentError,
    ArgumentsChecker,
    Call,
    CallState,
    CheckContext,
    CheckOptions,
    ConformanceError,
    Conversation,
    HistoryCheckpoint,
    Host,
    HostClosure,
    Result,
    SystemPrompt,
    ToolSpec,
    User,
    Wait,
    check_arguments,
    checkers_for,
    parse_arguments,
)

weather = ToolSpec(
    name="weather",
    description="Weather for a city.",
    json_schema={
        "type": "object",
        "properties": {"city": {"type": "string"}},
        "required": ["city"],
    },
)
system = SystemPrompt.build(tools=[weather])
prompt = Conversation(system=system, items=[User(payload="Weather in Zurich?")])
context = CheckContext(prefix=prompt)

## Raw arguments stay available

Malformed argument JSON is representable. The parser can return it to the harness, which decides the tool/error response. JSON parsing and schema validation are explicit operations.


In [ ]:
raw = Call(name="weather", counter=0, payload="not JSON")
assert raw.check(mode="profile").ok
try:
    parse_arguments(raw)
except ArgumentError as error:
    print(error)
assert not check_arguments(raw, weather).ok

output = Conversation(items=[raw, Wait()])
report = output.check(context=context)
assert not report.ok and report.complete
print([(finding.rule, str(finding.location)) for finding in report.errors])

valid = Call.build(name="weather", counter=0, arguments={"city": "Zurich"})
output = Conversation(items=[valid, Wait()])
report = output.check(context=context)
assert report.ok and report.complete
# Pending calls are allowed; a transcript need not end in Wait.
assert Conversation(system=system, items=[valid]).check().ok

## Checked loading and missing context

Checked loading/building returns a model only if the report is both OK and complete. Standalone full checks cannot assume tool declarations, prior counters, or waiting state.


In [ ]:
incomplete = valid.check(mode="full")
assert incomplete.ok and not incomplete.complete
try:
    Call.from_json(valid.to_json(), check="full")
except ConformanceError as error:
    assert error.report.ok and not error.report.complete

options = CheckOptions(mode="full", context=context)
assert Conversation.from_json(output.to_json(), check=options) == output
assert output.check(
    checkers=[*checkers_for("conversation"), ArgumentsChecker()],
    context=context,
).complete

## Retained history and host closures

A checkpoint describes the known state before a retained contiguous fragment. Unknown fields remain unavailable evidence. Host prose is never interpreted: the harness supplies explicit closure records, or `host_closures=()` to state that no host messages close calls.


In [ ]:
checkpoint = HistoryCheckpoint(
    system=system,
    next_call_counter=4,
    calls=(CallState(counter=3, name="weather"),),
    document_ids=frozenset(),
    waiting=True,
)
fragment = Conversation(
    items=[
        Result(name="weather", counter=3, payload="done"),
        Call.build(name="weather", counter=4, arguments={"city": "Bern"}),
    ]
)
report = fragment.check(context=CheckContext(history="partial", checkpoint=checkpoint))
assert report.ok and report.complete

closed = Conversation(
    system=system,
    items=[
        valid,
        Host(payload="NotCompleted: call counter 0 is closed."),
        Result(name="weather", counter=0, payload="late result"),
    ],
)
unknown_closure = closed.check()
assert unknown_closure.ok and not unknown_closure.complete
known_closure = closed.check(
    context=CheckContext(
        host_closures=(HostClosure(item=1, counter=0, name="weather"),),
    )
)
assert any(finding.rule == "conversation/result" for finding in known_closure.errors)
assert known_closure.complete